# 03 - Model Training

Build, train, and evaluate deep learning models for plant disease classification.

**Models:**
- ResNet50 (better accuracy)
- MobileNetV2 (faster, mobile-friendly)

**Tasks:**
1. Load data from `data/raw/`
2. Split: train/val/test (70/15/15)
3. Data augmentation
4. Train with PyTorch
5. Evaluate and save best model

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name()}')
else:
    print('Device: CPU')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# Configuration
DATA_ROOT = Path('..') / 'data' / 'raw'
MODEL_SAVE_DIR = Path('..') / 'models'
NUM_CLASSES = 10
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
SEED = 42

# Image preprocessing
IMG_SIZE = 224

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet stats
        std=[0.229, 0.224, 0.225]
    )
])

transform_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print(f'Dataset root: {DATA_ROOT}')
print(f'Model save dir: {MODEL_SAVE_DIR}')
print(f'Config: {NUM_CLASSES} classes, {IMG_SIZE}x{IMG_SIZE} images')

## 1. Load Dataset

In [ ]:
# Load full dataset
dataset_full = ImageFolder(root=str(DATA_ROOT), transform=transform_train)

print(f'Total images: {len(dataset_full)}')
print(f'Classes: {dataset_full.classes}')
print(f'Class to index: {dataset_full.class_to_idx}')

In [ ]:
from torch.utils.data import random_split

# Split: 70% train, 15% val, 15% test
train_size = int(0.7 * len(dataset_full))
val_size = int(0.15 * len(dataset_full))
test_size = len(dataset_full) - train_size - val_size

torch.manual_seed(SEED)
train_set, val_set, test_set = random_split(
    dataset_full, 
    [train_size, val_size, test_size]
)

# Apply different transforms for val/test
val_set.dataset.transform = transform_val
test_set.dataset.transform = transform_val

print(f'Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}')

In [ ]:
# Create dataloaders
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Batches - Train: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}')

## 2. Build Model

In [ ]:
import torchvision.models as models

# Load pretrained ResNet50
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# Replace final layer for 10 classes
model.fc = nn.Linear(2048, NUM_CLASSES)
model = model.to(device)

print(f'Model: ResNet50')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M')

## 3. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

print('Training setup complete')
print(f'Loss: CrossEntropyLoss')
print(f'Optimizer: AdamW (lr={LEARNING_RATE})')
print(f'Scheduler: CosineAnnealingLR')

In [ ]:
# Training loop (simplified)
# Full implementation in src/train.py

history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0

print(f'Training for {NUM_EPOCHS} epochs...')
print('(Simplified - see src/train.py for full implementation)')

## 4. Evaluate

In [ ]:
# Evaluation on test set
# Will compute: accuracy, precision, recall, F1-score, confusion matrix

print('Evaluation metrics:') print('- Accuracy')
print('- Precision (per class)')
print('- Recall (per class)')
print('- F1-score')
print('- Confusion Matrix')
print('- Classification Report')

## Note

Full training implementation will be in:
- `src/train.py` - Training script
- `src/evaluate.py` - Evaluation metrics

This notebook is a reference/skeleton for the training pipeline.